# 02 · Implied volatility surface — Options Surface Lab

**FR-11 / T-25.** Notebook 01 established what the panel *is*: a sparse cloud of marks, most
of them never traded, many of them wide enough that the mark is a guess. This notebook does
the next thing you are tempted to do with such a panel — push it through Black-Scholes and
read off a volatility — and then looks honestly at what came out.

The point is not the surface. The point is **where the inversion refuses, and why**, because
that is the boundary of what this data can support. Three questions:

1. Does the solver actually recover a vol we put in? (§2)
2. Which rows refuse, and for what reason? (§4)
3. How much does the constant rate — an assumption we chose, not a measurement — move the
   answer? (§5)

Everything numerical here comes from `options_surface_lab.option_surface_utils`. Nothing is
re-derived locally: a transform worth keeping graduates into the package with a test
(ARCHITECTURE §7), and the figure at the end is the same builder the published page uses.

In [1]:
import sys
from pathlib import Path

# repo root on sys.path whether the kernel starts in notebooks/ or the repo root
_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import plotly.express as px

from options_surface_lab.option_surface_utils import (
    DAYS_PER_YEAR,
    IV_BOUNDS,
    MARK_FIELD_DEFAULT,
    RISK_FREE_RATE,
    attach_implied_vol,
    attach_underlying,
    bs_price,
    flatten_lseg_options,
    implied_vol,
    iv_refusal,
    load_payload,
    pivot_trade_settle,
)

payload = load_payload()
wide = pivot_trade_settle(attach_underlying(flatten_lseg_options(payload["options"]), payload["stock"]))

print("source     :", "SYNTHETIC (invented)" if payload.get("synthetic") else f"real cache, {payload.get('fetched_at')}")
print("panel      :", f"{len(wide):,} contract-days", "|", wide["ric"].nunique(), "series", "|", wide["date"].nunique(), "dates")
print("dte range  :", int(wide["dte"].min()), "->", int(wide["dte"].max()), "days")
print("mark slot  :", MARK_FIELD_DEFAULT, "| rate r =", f"{RISK_FREE_RATE:.2%}", "| bracket:", IV_BOUNDS)

source     : real cache, 2026-08-30 11:11:14
panel      : 7,458 contract-days | 296 series | 53 dates
dte range  : 0 -> 64 days
mark slot  : MID_PRICE | rate r = 4.00% | bracket: (0.0001, 5.0)


## 1 · The model, and what it costs

Black-Scholes gives a price for a European option under a fixed volatility. Invert it and you
get "the volatility that would make this model reproduce this price." That number is only as
meaningful as the four things we had to assume to get it:

| Assumption | Reality | Why we accept it anyway |
|---|---|---|
| **European exercise** | These are American-style US listed equity options | Early exercise is worth roughly nothing on a non-dividend payer; it biases deep-ITM puts most |
| **No dividends** | UUUU pays none in the window | Actually true here — the cheapest assumption on the list |
| **act/365** | Calendar days, not trading days | A convention; act/252 would raise every vol by ~√(365/252) ≈ 1.2× uniformly |
| **Constant `r` = 4.00%** | The curve moves daily | PRD OQ-2, closed by the PO 2026-09-04. §5 measures what it costs |

And one more, which is the one this whole lab is about: **the price we invert is itself
derived.** There is no settlement price for US listed equity options (notebook 01 §10b), so
`MARK` is the closing NBBO midpoint. Notebook 01 measured the median bid-ask at ~20% of the
mark — so before any modelling, the input already carries a ±10% band. An implied vol built
on it inherits that, and nothing downstream can remove it.

That is why the shipped figure is captioned *derived, not observed* and why it says on its
face that it is not a tradable price.

## 2 · Does the inversion actually work?

The round trip, run in the open: pick a real contract-day from the panel, take its mark,
invert it, then re-price at the recovered vol and check we land back on the mark we started
from. `tests/test_iv.py` asserts this over a grid of hand-built cases; here it is
on a row from the actual pull, so the arithmetic is visible rather than merely green.

In [2]:
# A real, well-behaved row: near the money, a few weeks out, both a mark and a print.
candidates = wide[
    wide["MARK"].notna() & wide["TRDPRC_1"].notna()
    & wide["dte"].between(14, 45) & wide["moneyness"].between(0.95, 1.05)
].sort_values("date")
row = candidates.iloc[len(candidates) // 2]

t_years = row["dte"] / DAYS_PER_YEAR
sigma = implied_vol(row["MARK"], row["spot"], row["strike"], t_years, RISK_FREE_RATE, row["cp"])
repriced = bs_price(row["spot"], row["strike"], t_years, sigma, RISK_FREE_RATE, row["cp"])

print(f"{row['ric']}  ({row['cp']})  as-of {row['date'].date()}")
print(f"  spot   S = ${row['spot']:.2f}    strike K = ${row['strike']:.2f}    dte = {int(row['dte'])}d  (T = {t_years:.4f}y)")
print(f"  mark       = ${row['MARK']:.4f}   ({MARK_FIELD_DEFAULT})")
print(f"  implied vol= {sigma:.4%}")
print(f"  re-priced  = ${repriced:.4f}   <- must equal the mark")
print(f"  round-trip error = ${abs(repriced - row['MARK']):.2e}")
print()
print(f"  for scale: the last trade that day was ${row['TRDPRC_1']:.4f}, and the bid-ask was "
      f"${row['spread']:.2f} ({row['spread_pct']:.0f}% of the mark)")

UUUUT142601350.U^H26  (P)  as-of 2026-07-02
  spot   S = $13.81    strike K = $13.50    dte = 43d  (T = 0.1178y)
  mark       = $1.2850   (MID_PRICE)
  implied vol= 78.7329%
  re-priced  = $1.2850   <- must equal the mark
  round-trip error = $7.84e-14

  for scale: the last trade that day was $1.2800, and the bid-ask was $0.25 (19% of the mark)


In [3]:
# The same row, the other way round: what the price curve looks like as a function of vol.
# The inversion is just "read this curve backwards" — it works because the curve is monotone,
# and it gets hard exactly where the curve goes flat.
grid = np.linspace(0.05, 3.0, 120)
curve = pd.DataFrame({
    "sigma": grid,
    "model price": [bs_price(row["spot"], row["strike"], t_years, s, RISK_FREE_RATE, row["cp"]) for s in grid],
})
fig = px.line(curve, x="sigma", y="model price",
              title=f"{row['ric']}: model price vs vol — the curve the solver inverts")
fig.add_hline(y=row["MARK"], line_dash="dot", annotation_text=f"the mark ${row['MARK']:.3f}")
fig.add_vline(x=sigma, line_dash="dot", annotation_text=f"σ = {sigma:.1%}")
fig

## 3 · The whole panel

`attach_implied_vol` inverts every contract-day that can be inverted and leaves NaN where it
cannot. NaN is the design, not a failure: FR-11 requires degenerate inputs to produce **gaps
rather than errors or absurd vols** (SYSTEM-SPEC §12), and a hole in the surface is a true
statement about what the data supports.

In [4]:
iv = attach_implied_vol(wide)
solved = iv["iv"].notna()

print(f"contract-days      : {len(iv):,}")
print(f"inverted           : {solved.sum():,}  ({100 * solved.mean():.1f}%)")
print(f"refused (gaps)     : {(~solved).sum():,}  ({100 * (~solved).mean():.1f}%)")
print()
print((iv.loc[solved, "iv"] * 100).describe(percentiles=[.05, .25, .5, .75, .95]).round(1).to_string())

contract-days      : 7,458
inverted           : 6,275  (84.1%)
refused (gaps)     : 1,183  (15.9%)

count    6275.0
mean       94.4
std        36.3
min        37.4
5%         72.7
25%        80.4
50%        85.8
75%        93.3
95%       145.7
max       493.6


In [5]:
px.histogram(
    iv.loc[solved].assign(**{"implied vol (%)": iv.loc[solved, "iv"] * 100}),
    x="implied vol (%)", color="cp", nbins=80, barmode="overlay", opacity=0.65,
    title="Inverted vols across the panel — calls and puts",
)

## 4 · Where the inversion degenerates

This is the section T-25 exists for. `iv_refusal` names, for a single row, the reason the
solver will not touch it — and returns `None` when the row is invertible. Applying it across
the panel turns "84% inverted" into an account of *what the other 16% are*.

The categories are not bugs. Each one is a place where the model has nothing to say:

- **expiry day** — `dte = 0`, so `T = 0`. There is no time for a volatility to act over; the
  option is worth its intrinsic value and no vol reproduces anything else.
- **at or below intrinsic** — a European price below its own floor implies a negative time
  value. On American contracts this happens for real (deep-ITM options trade at parity), and
  a stale or one-sided midpoint produces it too.
- **above the no-arbitrage cap** — a call worth more than the stock, or a put worth more than
  its discounted strike. Almost always a bad quote.
- **no mark / no underlying close** — nothing to invert, or nothing to invert it against.
- **bracket miss** — analytically fine, but no vol in `[0.01%, 500%]` reproduces the price.
  These are penny quotes on far wings, where "the implied vol" is arbitrarily large and
  reporting it would be arithmetic dressed as information.

In [6]:
reasons = [
    iv_refusal(r.MARK, r.spot, r.strike, r.dte / DAYS_PER_YEAR, RISK_FREE_RATE, r.cp)
    for r in iv.itertuples()
]
iv = iv.assign(refusal=reasons)
# A row the analytic checks pass but the solver still could not bracket.
iv.loc[iv["refusal"].isna() & iv["iv"].isna(), "refusal"] = "bracket miss — no vol in [0.01%, 500%] fits"
iv["refusal"] = iv["refusal"].fillna("inverted")

breakdown = (
    iv["refusal"].value_counts().rename("contract-days").to_frame()
    .assign(**{"% of panel": lambda d: (100 * d["contract-days"] / len(iv)).round(2)})
)
breakdown

,contract-days,% of panel
refusal,,
inverted,6275,84.14
no mark to invert,586,7.86
expiry day — no time left to carry a vol,296,3.97
at or below intrinsic — no time value to explain,293,3.93
"bracket miss — no vol in [0.01%, 500%] fits",8,0.11


In [7]:
px.bar(
    breakdown.drop(index="inverted").reset_index(names="reason").sort_values("contract-days"),
    x="contract-days", y="reason", orientation="h", text="contract-days",
    title="Why the solver refused — every one of these is a hole in the surface, never a filled-in vol",
)

### 4a · The near-expiry cliff

Refusals are not spread evenly. As `dte` falls, time value collapses toward zero and the
price stops carrying information about volatility — so the same $0.01 of quote noise swings
the implied vol further and further, until the price falls through the intrinsic floor and
the solver gives up entirely.

The last date of an expired-weeklies panel is the extreme case (notebook 01 / T-36: one
expiry alive, everything at or near expiry), which is exactly why the app opens on the
busiest date rather than the last one.

In [8]:
by_dte = (
    iv.assign(inverted=iv["iv"].notna())
    .groupby("dte")
    .agg(contract_days=("inverted", "size"), inverted=("inverted", "mean"), median_iv=("iv", "median"))
    .reset_index()
)
by_dte["inverted %"] = 100 * by_dte["inverted"]
by_dte["median iv %"] = 100 * by_dte["median_iv"]

px.line(
    by_dte, x="dte", y=["inverted %", "median iv %"], markers=True,
    title="Inversion success and vol level by days to expiry — the cliff is at the short end",
)

In [9]:
# The same thing said as a table, short end first.
by_dte[by_dte["dte"] <= 10][["dte", "contract_days", "inverted %", "median iv %"]].round(1)

,dte,contract_days,inverted %,median iv %
0,0,296,0.0,NaN
1,1,296,56.8,151.1
2,2,296,57.1,115.6
3,3,296,68.6,111.7
4,4,296,75.3,99.8
5,7,202,67.8,84.4
6,8,268,76.9,84.0
7,9,268,78.4,86.6
8,10,268,83.6,87.3


### 4b · Sub-intrinsic marks

The largest refusal category deserves a look rather than a shrug. A mark below the European
intrinsic floor is not necessarily a bad quote — it is what deep-in-the-money **American**
options do, because the right to exercise early puts a floor under them that our model does
not include. Sort them by moneyness and the pattern is immediate.

In [10]:
sub = iv[iv["refusal"].str.startswith("at or below intrinsic")].copy()
sub["european floor"] = [
    max(r.spot - r.strike * np.exp(-RISK_FREE_RATE * r.dte / DAYS_PER_YEAR), 0.0) if r.cp == "C"
    else max(r.strike * np.exp(-RISK_FREE_RATE * r.dte / DAYS_PER_YEAR) - r.spot, 0.0)
    for r in sub.itertuples()
]
sub["mark - floor"] = sub["MARK"] - sub["european floor"]

print(f"{len(sub):,} contract-days sit at or below the European floor")
print(sub["moneyness"].describe(percentiles=[.1, .5, .9]).round(3).to_string())
print()
print("deep in the money on both sides — exactly where early exercise bites:")
sub.groupby("cp")["moneyness"].describe()[["count", "25%", "50%", "75%"]].round(3)

293 contract-days sit at or below the European floor
count    293.000
mean       1.179
std        0.296
min        0.634
10%        0.765
50%        1.248
90%        1.533
max        1.723

deep in the money on both sides — exactly where early exercise bites:


,count,25%,50%,75%
cp,,,,
C,98.0,0.754,0.802,0.846
P,195.0,1.250,1.362,1.471


In [11]:
px.scatter(
    sub, x="moneyness", y="mark - floor", color="cp", opacity=0.55,
    labels={"moneyness": "K / S", "mark - floor": "mark − European floor ($)"},
    title="Sub-intrinsic marks: calls fail deep ITM (K/S < 1), puts deep ITM (K/S > 1)",
).add_hline(y=0, line_dash="dot")

## 5 · How much does the rate assumption cost?

PRD **OQ-2** was closed by the PO on 2026-09-04 at `r = 4.00%`, cited as a short T-bill yield.
The PRD's own framing is that "the writing-down matters more than the number" — so this
section checks whether that is true here rather than assuming it.

It is *mostly* true, and the exception is the interesting part.

In [12]:
rates = [0.00, 0.02, 0.04, 0.06]
vols = {f"r = {r:.0%}": attach_implied_vol(wide, rate=r)["iv"] for r in rates}
sens = pd.DataFrame(vols)
common = sens.notna().all(axis=1)

print(f"rows invertible at every rate: {common.sum():,}")
print()
print("median inverted vol, by assumed rate:")
print((sens[common].median() * 100).round(2).to_string())
print()
shift = (sens.loc[common, "r = 4%"] - sens.loc[common, "r = 0%"]).abs() * 100
print("|vol(4%) - vol(0%)| over rows invertible at ALL FOUR rates, in vol points:")
print(shift.describe(percentiles=[.5, .9, .95, .99]).round(2).to_string())

# The pair the docs quote: rows invertible at 0% and 4% only. Wider set, heavier tail.
pair = sens[["r = 0%", "r = 4%"]].dropna()
pair_shift = (pair["r = 4%"] - pair["r = 0%"]).abs() * 100
print()
print(f"over the {len(pair):,} rows invertible at 0% and 4% (the figure the docs quote):")
print(f"  median {pair_shift.median():.2f}  p95 {pair_shift.quantile(.95):.2f}  "
      f"max {pair_shift.max():.2f}  vol points")

rows invertible at every rate: 6,223

median inverted vol, by assumed rate:
r = 0%    85.28
r = 2%    85.53
r = 4%    85.83
r = 6%    86.00

|vol(4%) - vol(0%)| over rows invertible at ALL FOUR rates, in vol points:
count    6223.00
mean        1.92
std         2.01
min         0.09
50%         1.28
90%         4.02
95%         5.61
99%         9.95
max        21.74

over the 6,229 rows invertible at 0% and 4% (the figure the docs quote):
  median 1.28  p95 5.69  max 24.96  vol points


Read that carefully before concluding the rate is free.

The **median** row moves by about a vol point on a panel whose median vol is ~86% — a ~1.5%
relative change, genuinely immaterial. But the **tail does not**: the 95th percentile moves
several vol points and the worst rows move tens. Averaging over the panel would have hidden
that, which is why the shift is plotted against moneyness rather than summarised into one
number.

**Quote the row set with the figure.** The cell above restricts to rows invertible at *all
four* probed rates; the docs quote the narrower and more natural pair, rows invertible at
0% and 4% (6,229 of them: median 1.28, p95 5.69, max 24.96). The two differ, and the first
version of this work quoted one subset's tail under the other's sentence — which is exactly
the kind of error a sensitivity analysis exists to prevent, so it is recorded here rather
than quietly corrected.

The mechanism: `r` enters only through the discounted strike `K·e^(−rT)`, which moves the
intrinsic floor. Far from the money that floor is nowhere near the price and nothing happens.
Deep in the money the price sits just above the floor with almost no time value — vega is
tiny — so a small shift in the floor demands a large shift in vol to compensate. The same
flatness that makes §4b's rows refuse outright makes their neighbours rate-sensitive.

In [13]:
sens_df = wide.loc[common, ["cp", "moneyness", "dte"]].assign(**{"vol-point shift": shift})
px.scatter(
    sens_df, x="moneyness", y="vol-point shift", color="cp", opacity=0.4,
    labels={"moneyness": "K / S", "vol-point shift": "|vol(4%) − vol(0%)|  (vol points)"},
    title="The rate is free at the money and expensive deep in it",
)

In [14]:
# Same story, tabulated by moneyness band — the shape to remember, not the exact numbers.
band = pd.cut(sens_df["moneyness"], [0, 0.8, 0.95, 1.05, 1.25, np.inf],
              labels=["deep ITM call side (<0.80)", "0.80–0.95", "near the money", "1.05–1.25", "deep (>1.25)"])
sens_df.assign(band=band).groupby("band", observed=True)["vol-point shift"].describe()[
    ["count", "50%", "max"]
].round(2)

,count,50%,max
band,,,
deep ITM call side (<0.80),457.0,1.77,11.09
0.80–0.95,1393.0,1.33,10.11
near the money,1191.0,1.16,2.58
1.05–1.25,2174.0,1.22,10.71
deep (>1.25),1008.0,3.46,21.74


**Conclusion for the page.** `r = 4.00%` is defensible and its typical cost is about a vol
point. It is printed on the figure — not because the number is delicate, but because a reader
cannot tell which rows are in the insensitive middle and which are in the tail unless they
know what was assumed. That is the whole argument for writing assumptions down.

## 6 · The figure as it ships

Same builder the Reflex app and the published page call, so what appears here is what a
grader sees.

It is a **2D smile, not a 3D cloud**: the same numbers plotted over (K/S, DTE, σ) in three
dimensions read as scattered points with no discernible message (PO, 2026-09-04). Flattened
to one curve per expiry they have a shape you can name — and the thing that shape shows is
below.

It is also **not an interpolated sheet**. The hero surface already demonstrates what
interpolation does to a sparse cloud; doing it again on top of a model output would be
smoothing a guess over a guess. A break in a line is a strike the solver refused, drawn as a
break precisely so the line does not bridge it.

In [15]:
from options_surface_lab.option_surface_plot import iv_smile_figure

asof = wide.groupby(wide["date"].dt.normalize()).size().idxmax()   # the busiest date, as the app opens
print("as-of:", pd.Timestamp(asof).date())
iv_smile_figure(wide, asof, ticker=payload["ticker"])

as-of: 2026-07-10


In [16]:
# The same date under FR-10's other ruler. A change of units on one axis, nothing else:
# the vols, the colours and the trace names are identical (asserted in test_app_figures.py).
iv_smile_figure(wide, asof, ticker=payload["ticker"], x_mode="strike")

In [17]:
slice_ = iv[(iv["date"] == pd.Timestamp(asof)) & iv["iv"].notna()].copy()
slice_["implied vol (%)"] = slice_["iv"] * 100

# What the 2D form made visible and the 3D form hid: near-dated smiles are STEEP and
# far-dated ones are nearly flat. That difference is the term structure of the smile.
span = (
    slice_.groupby(slice_["expiry"].dt.date)["implied vol (%)"]
    .agg(["count", "min", "max"])
    .assign(**{"span (vol pts)": lambda d: (d["max"] - d["min"]).round(1)})
    .round(1)
)
print("near-dated expiry first — the span collapses as you go out:")
print(span.to_string())
print()

# The raw points behind the curves, for anyone who wants them.
px.scatter(
    slice_.sort_values("moneyness"), x="moneyness", y="implied vol (%)",
    color=slice_["expiry"].dt.strftime("%b %d"), symbol="cp", opacity=0.85,
    labels={"moneyness": "K / S", "color": "expiry"},
    title=f"Smile cross-sections, {pd.Timestamp(asof).date()} — one expiry per colour",
)

near-dated expiry first — the span collapses as you go out:
            count   min    max  span (vol pts)
expiry                                        
2026-07-17     24  48.0  129.8            81.8
2026-07-24     30  61.2  131.8            70.6
2026-07-31     32  69.5  110.3            40.7
2026-08-07     34  70.0   91.5            21.5
2026-08-14     34  72.2   86.8            14.6
2026-08-21     16  73.8   87.1            13.4



## 7 · What this hands to Assignment 1.2

1.2 asks for simulated fills off a volatility surface. Three things from this notebook set the
boundary on how much that can be trusted:

- **The surface has holes, and they are not random.** They cluster at short dte and deep in
  the money (§4a, §4b) — which is precisely where an execution simulator most wants a number.
  Any 1.2 surface fit has to decide what to do there, and "interpolate" is a decision, not a
  default.
- **The input is a midpoint, not a fill.** The median bid-ask in this panel is ~20% of the
  mark. A vol inverted from the mid implies a price you could not actually trade at; filling
  at mid is optimistic by roughly half the spread, and that error is much larger than
  anything the rate assumption contributes.
- **Sensitivity is concentrated, not uniform.** §5's tail says a single summary statistic
  about model error will mislead. The honest version reports error *by regime*.

The one thing this notebook does **not** claim is that any number here is tradable. That
caveat is on the figure, on the page, and here.